# 🛒 BİRLİKTELİK KURALI (ASSOCIATION RULE MINING - APRIORI)

> **🎯 AMAÇ:** "Bunu alan onu da aldı" ilişkilerini çözmek.  
> **📥 GİRDİ:** X_train (Veri Ön İşleme'den gelen - Genelde Liste Listesi olması gerekir)  
> **📤 ÇIKTI:** Kurallar (Rules), Support, Confidence, Lift değerleri  

### ⏱️ NE ZAMAN KULLANILIR?
- Market sepeti analizi
- Öneri sistemleri (Netflix, Amazon vb.)
- Promosyon tasarımı

### ⚠️ UYARI
Bu dosyayı çalıştırmadan **ÖNCE** şunlardan emin olun:
1. **`apyori.py`** dosyası bu notebook ile **AYNI** klasörde olmalı (GitHub'dan indirdiyseniz yanındadır).
2. Veri ön işleme yapıldıysa `X_train` değişkeni bellekte olmalı.

```python
# Eğer .py dosyanız varsa:
%run Veri_On_Isleme.py
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# apyori.py dosyası aynı klasörde olduğu için direkt import edilebilir
from apyori import apriori

print("=" * 40)
print("🛒 APRIORI ANALİZİ BAŞLATILIYOR")
print("=" * 40)

In [ ]:
# 1. VERİ HAZIRLIĞI (LIST OF LISTS)
# ---------------------------------------------------
# Apriori algoritması Pandas DataFrame değil, "Liste İçinde Liste" formatı ister.
# Örnek: [['Ekmek', 'Süt'], ['Bez', 'Ekmek'], ...]

print("\n🔄 Veri (X_train) Apriori formatına çevriliyor...")

transactions = []

# X_train'in Numpy Array veya DataFrame olmasına göre işlem yap
# (Veri_On_Isleme'den genelde Numpy array gelir)
data_source = X_train # Bellekteki veri

rows = len(data_source)
cols = len(data_source[0]) if hasattr(data_source, '__getitem__') else data_source.shape[1]

for i in range(rows):
    # Her satırdaki elemanları string'e çevirip listeye ekle
    # 'nan' değerleri almamak için filtreleme yapılabilir
    transaction = [str(data_source[i, j]) for j in range(cols) if str(data_source[i, j]) != 'nan']
    transactions.append(transaction)

print(f"✅ {len(transactions)} işlem (sepet) hazırlandı.")
print(f"👀 Örnek Sepet: {transactions[0]}")

In [ ]:
# 2. KURALLARIN OLUŞTURULMASI
# ---------------------------------------------------
# <--- LÜTFEN BURAYI DÜZENLEYİN: Eşik değerleri projenize göre ayarlayın
# 
# 1. min_support (Destek): Bir ürünün/kombinasyonun toplam işlem içindeki oranı.
#    - Örn: 0.01 -> %1 (100 işlemde en az 1 kere geçmiş olmalı)
#    - Ne zaman artırılır? Çok fazla ve gereksiz kural çıkıyorsa.
#    - Ne zaman azaltılır? Hiç kural çıkmıyorsa.
#
# 2. min_confidence (Güven): X'i alanın Y'yi alma olasılığı.
#    - Örn: 0.2 -> %20 (X alanların en az %20'si Y'yi de almış olmalı)
#    - Güvenilir kurallar için bu değeri yüksek tutun (0.5+ gibi).
#
# 3. min_lift (Lift - İlişki Gücü): Kuralın şansa bağlı olup olmadığını gösterir.
#    - Lift = 1 -> İlişki yok (Bağımsız).
#    - Lift > 1 -> Pozitif ilişki (Bunu alan onu da alıyor).
#    - Lift < 1 -> Negatif ilişki (Bunu alan onu almıyor).
#    - Genelde 3 ve üzeri çok güçlü kabul edilir.
#
# 4. min_length: Kuralda en az kaç ürün olsun? (Genelde 2: En az biri diğerini tetiklesin)
min_support_val = 0.01
min_confidence_val = 0.2
min_lift_val = 3.0
min_length_val = 2

print("\n🚀 Kurallar çıkartılıyor...")

rules = apriori(transactions, 
                min_support=min_support_val, 
                min_confidence=min_confidence_val, 
                min_lift=min_lift_val, 
                min_length=min_length_val)

# Sonuçları listeye çevirme
results = list(rules)

print(f"✅ Toplam {len(results)} kural bulundu.")

In [ ]:
# 3. SONUÇLARI GÖRSELLEŞTİRME / YAZDIRMA
# ---------------------------------------------------
# Nedir bu fonksiyon? 
# 'apyori' kütüphanesinin çıktısı (rules) çok karmaşık bir yapıdadır.
# Bu fonksiyon, o karmaşık yapıyı (RelationRecord) düz bir listeye çevirir,
# böylece Pandas ile Excel tablosu gibi görebiliriz.

print("\n📊 EN GÜÇLÜ İLİŞKİLER:\n")

def inspect(results):
    visited_base = []
    visited_add = []
    supports = []
    confidences = []
    lifts = []
    
    for result in results:
        # Result değişkeni şuna benzer: [items, support, [ordered_stats...]]
        items = list(result[0])
        if len(items) > 1: # Sadece 2 veya daha fazla ürünlü kuralları al
            # İlişkiyi parçala: (Ürün 1) -> (Ürün 2)
            base = items[0]
            add = items[1]
            
            visited_base.append(base)
            visited_add.append(add)
            supports.append(result[1])
            confidences.append(result[2][0][2])
            lifts.append(result[2][0][3])
            
    return list(zip(visited_base, visited_add, supports, confidences, lifts))

try:
    table_data = inspect(results)
    results_df = pd.DataFrame(table_data, columns=['Ürün 1', 'Ürün 2', 'Support', 'Confidence', 'Lift'])
    
    # Lift değerine göre sırala (En güçlü ilişki en üstte görünsün diye)
    results_df = results_df.nlargest(n=10, columns='Lift')
    
    print(results_df.to_string(index=False))
    
except Exception as e:
    print(f"⚠️ Tablo oluşturulurken hata: {e}")
    print("Ham sonuçlar:")
    for i in results[:5]:
        print(i)